# Cài đặt thư viện cần thiết

In [1]:
# Cài đặt các thư viện cần thiết cho quá trình tiền xử lý và mô hình
!pip install -q pyvi emoji transformers scikit-learn openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.6 MB/s eta 0:00:00


# Khai báo thư viện và kiểm tra GPU

In [2]:
import pandas as pd
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_scheduler
from torch.optim import AdamW
from pyvi.ViTokenizer import tokenize 
import emoji
import re
from tqdm import tqdm
import time
from sklearn.metrics import f1_score, classification_report

# Kiểm tra thiết bị (GPU/CPU)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('Thiết bị đang sử dụng:', device)

# Tạo sẵn thư mục lưu model và report trên Kaggle để tránh lỗi FileNotFoundError
os.makedirs('/kaggle/working/saved_models', exist_ok=True)
os.makedirs('/kaggle/working/reports', exist_ok=True)

Thiết bị đang sử dụng: cuda:0


# Cấu hình đường dẫn và Tải Dữ liệu (Dictionaries & Dataset) 

In [3]:
import os
import json
import pandas as pd
import subprocess

# =====================================================================
# 1. Tải repo ViGoEmotions để lấy TOÀN BỘ file (Docs + Corpus)
# =====================================================================
REPO_URL = "https://github.com/ricardo-tran/ViGoEmotions.git"
REPO_DIR = "/kaggle/working/ViGoEmotions_Original"

if not os.path.exists(REPO_DIR):
    print("--- 1. Đang tải mã nguồn ViGoEmotions từ GitHub... ---")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR])
else:
    print("--- 1. Thư mục repo GitHub đã tồn tại ---")

# =====================================================================
# 2. Định nghĩa đường dẫn trỏ thẳng vào thư mục GitHub vừa tải
# =====================================================================
DOCS_PATH = os.path.join(REPO_DIR, 'model', 'docs')
CORPUS_PATH = os.path.join(REPO_DIR, 'corpus')

# =====================================================================
# 3. Tải Dictionaries (Từ điển)
# =====================================================================
print("\n--- 2. Đang tải Dictionaries ---")
with open(os.path.join(DOCS_PATH, 'patterns.json'), 'r', encoding='utf-8') as f:
    pattern_dict = json.load(f)
with open(os.path.join(DOCS_PATH, 'emojis.json'), 'r', encoding='utf-8') as f:
    emoji_dict = json.load(f)
with open(os.path.join(DOCS_PATH, 'teencode4.txt'), 'r', encoding='utf-8') as f:
    teen_dict = {line.split('\t')[0]: line.split('\t')[1] for line in f.read().split('\n') if line.strip() and len(line.split('\t')) >= 2}
print("✅ Tải Dictionaries thành công!")

# =====================================================================
# 4. Tải Dataset (Đọc từ dataset_V1.xlsx như code gốc)
# =====================================================================
print("\n--- 3. Đang tải Dataset từ dataset_V1.xlsx ---")
excel_path = os.path.join(CORPUS_PATH, 'dataset_V1.xlsx')

excel_file = pd.ExcelFile(excel_path)
if 'train' in excel_file.sheet_names:
    train_df = pd.read_excel(excel_file, sheet_name='train')
    val_df = pd.read_excel(excel_file, sheet_name='val')
    test_df = pd.read_excel(excel_file, sheet_name='test')
else:
    df = pd.read_excel(excel_file, sheet_name='Sheet1')
    train_df = df[df['set'] == 'train']
    val_df = df[df['set'] == 'val']
    test_df = df[df['set'] == 'test']

print(f"✅ Kích thước tập Train: {train_df.shape} | Val: {val_df.shape} | Test: {test_df.shape}")

--- 1. Đang tải mã nguồn ViGoEmotions từ GitHub... ---


Cloning into '/kaggle/working/ViGoEmotions_Original'...



--- 2. Đang tải Dictionaries ---
✅ Tải Dictionaries thành công!

--- 3. Đang tải Dataset từ dataset_V1.xlsx ---
✅ Kích thước tập Train: (16531, 3) | Val: (2066, 3) | Test: (2067, 3)


In [4]:
!find /kaggle/working/ViGoEmotions_Original -maxdepth 3 -type f

/kaggle/working/ViGoEmotions_Original/.git/HEAD
/kaggle/working/ViGoEmotions_Original/.git/info/exclude
/kaggle/working/ViGoEmotions_Original/.git/config
/kaggle/working/ViGoEmotions_Original/.git/logs/HEAD
/kaggle/working/ViGoEmotions_Original/.git/description
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-merge-commit.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-applypatch.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/commit-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/update.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-receive.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-rebase.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/pre-push.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/post-update.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/prepare-commit-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/applypatch-msg.sample
/kaggle/working/ViGoEmotions_Original/.git/hooks/f

# Tiền xử lý văn bản (S2 Preprocessing)

In [5]:
def clean_text(text):
    text = text.lower()
    # 1. Normalize pattern (:)))) -> :)))
    for pattern, replacement in pattern_dict.items():
        text = re.sub(pattern=pattern, repl=replacement, string=text)
    
    # 2. Remove duplicate chars
    result = []
    prev_char = None
    for char in text:
        if char.isalpha() and prev_char == char: continue
        prev_char = char
        result.append(char)
    text = ''.join(result)
    
    # 3. Remove duplicate emojis
    result, prev_emoji = [], None
    for char in text:
        if char in emoji.EMOJI_DATA:
            if char == prev_emoji: continue
            prev_emoji = char
        else:
            prev_emoji = None
        result.append(char)
    text = ''.join(result)
    
    # 4. Replace teencode
    for old_word, new_word in teen_dict.items():
        pattern = re.compile(r'\b{}\b'.format(re.escape(old_word)))
        text = pattern.sub(new_word, text)
        
    # 5. Replace emojis
    for emoji_char, replacement in emoji_dict.items():
        text = text.replace(emoji_char, ' ' + replacement + ' ')
        
    # 6. Formatting punctuation
    text = re.sub(r'(?<![.,!?;:])\n', r'. ', text)  
    text = re.sub(r'\n([.,!?;:])?', r' \1', text)  
    text = re.sub(r'([.,!?;:])', r' \1 ', text)  
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Đang áp dụng tiền xử lý cho toàn bộ Dataset...")
for df in [train_df, val_df, test_df]:
    df['text'] = df['text'].apply(clean_text)
print("✅ Tiền xử lý hoàn tất!")

Đang áp dụng tiền xử lý cho toàn bộ Dataset...
✅ Tiền xử lý hoàn tất!


# Load Labels và Mã hóa One-hot (Label Encoding)

In [6]:
with open(os.path.join(DOCS_PATH, 'label_dict.json'), 'r', encoding='utf-8') as f:
    label_dict = json.load(f)

label_to_idx = {label: int(idx) for idx, label in label_dict.items()}

def encode_labels(label_str, label_dict):
    labels = label_str.replace('[', '').replace(']', '').replace("'", '').replace('"', '').split(',')
    labels = [label.strip() for label in labels]
    label_vector = np.zeros(len(label_dict), dtype=int)
    
    if labels[0].isnumeric(): 
        labels = [int(label) for label in labels]
        for idx in label_dict.values():
            if idx in labels: label_vector[idx] = 1
    else: 
        for label, idx in label_dict.items():
            if label in labels: label_vector[idx] = 1
    return label_vector

train_texts, train_labels = train_df['text'].tolist(), [encode_labels(lbl, label_to_idx) for lbl in train_df['labels']]
val_texts, val_labels = val_df['text'].tolist(), [encode_labels(lbl, label_to_idx) for lbl in val_df['labels']]
test_texts, test_labels = test_df['text'].tolist(), [encode_labels(lbl, label_to_idx) for lbl in test_df['labels']]
print("✅ Đã mã hóa nhãn thành công!")

✅ Đã mã hóa nhãn thành công!


# Khởi tạo Tokenizer và DataLoader

In [7]:
# Chọn model PhoBERT
model_type = 'phobert'
model_name = 'vinai/phobert-base-v2' 
max_len = 200

tokenizer = AutoTokenizer.from_pretrained(model_name)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        # Sửa thành torch.float để tương thích tốt nhất với PyTorch đời mới
        self.labels = torch.tensor(labels, dtype=torch.float)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): 
        return len(self.labels)

    def __getitem__(self, idx):
        text = tokenize(self.texts[idx]) # Pyvi Tokenizer
        
        # ĐÃ SỬA Ở ĐÂY: Gọi trực tiếp tokenizer thay vì dùng .encode_plus()
        encoding = self.tokenizer(
            text, 
            truncation=True, 
            add_special_tokens=True, 
            max_length=self.max_len,
            padding='max_length', 
            return_attention_mask=True, 
            return_tensors='pt'
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': self.labels[idx],
        }

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer, max_len)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer, max_len)
test_dataset = SentimentDataset(test_texts, test_labels, tokenizer, max_len)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
print("✅ DataLoader sẵn sàng!")

config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ DataLoader sẵn sàng!


/tmp/ipykernel_23/1152848740.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  self.labels = torch.tensor(labels, dtype=torch.float)


# Cấu hình

In [8]:
class ModelSentimentClassifier(nn.Module):
    def __init__(self, n_classes, model_name, model_type):
        super(ModelSentimentClassifier, self).__init__()
        self.model_type = model_type
        config = AutoConfig.from_pretrained(model_name, hidden_dropout_prob=0.1, attention_probs_dropout_prob=0.1)
        self.backbone = AutoModel.from_pretrained(model_name, config=config)
        self.drop = nn.Dropout(p=0.2)
        self.fc = nn.Linear(self.backbone.config.hidden_size, n_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        if 'bartpho' in self.model_type:
            output = outputs.last_hidden_state[:, 0, :] 
        else:
            output = outputs.pooler_output
        x = self.drop(output)
        return {'logits': self.fc(x)}

model = ModelSentimentClassifier(n_classes=len(label_dict), model_name=model_name, model_type=model_type).to(device)

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: vinai/phobert-base-v2
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]

# Thiết lập Hàm Huấn luyện & Đánh giá

In [9]:
EPOCHS = 12
optimizer = AdamW(model.parameters(), lr=5e-5)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=len(train_loader), num_training_steps=len(train_loader)*EPOCHS)  

# Xử lý mất cân bằng dữ liệu với pos_weight
label_counts = np.sum(train_labels, axis=0)
pos_weight = torch.tensor([(len(train_labels) - count) / count for count in label_counts]).to(device)
loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def train_eval_loop(model, data_loader, is_train=True):
    if is_train: model.train()
    else: model.eval()
    
    losses, all_targets, all_preds = [], [], []
    
    with torch.set_grad_enabled(is_train):
        for data in data_loader:
            input_ids, attention_mask, targets = data['input_ids'].to(device), data['attention_mask'].to(device), data['targets'].to(device)
            if is_train: optimizer.zero_grad()
            
            logits = model(input_ids, attention_mask)['logits']
            loss = loss_fn(logits, targets)
            losses.append(loss.item())
            
            if is_train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                lr_scheduler.step()

            preds = (torch.sigmoid(logits) >= 0.5).int()
            all_targets.append(targets.cpu().numpy())
            all_preds.append(preds.cpu().numpy())

    all_targets, all_preds = np.vstack(all_targets), np.vstack(all_preds)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    return np.mean(losses), macro_f1

# train loop

In [10]:
best_f1 = 0
start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_f1 = train_eval_loop(model, train_loader, is_train=True)
    print(f"Train Loss: {train_loss:.4f} | Macro F1: {train_f1:.4f}")
    
    val_loss, val_f1 = train_eval_loop(model, val_loader, is_train=False)
    print(f"Val Loss: {val_loss:.4f} | Macro F1: {val_f1:.4f}")
    
    if val_f1 > best_f1:
        print('⭐ Lưu mô hình tốt nhất...')
        torch.save(model.state_dict(), f'/kaggle/working/saved_models/{model_type}_best.pth')
        best_f1 = val_f1

print(f'\nHoàn tất đào tạo! Tổng thời gian: {(time.time() - start_time):.2f} giây')


Epoch 1/12
Train Loss: 1.0921 | Macro F1: 0.2177
Val Loss: 0.8597 | Macro F1: 0.3259
⭐ Lưu mô hình tốt nhất...

Epoch 2/12
Train Loss: 0.7356 | Macro F1: 0.3852
Val Loss: 0.6782 | Macro F1: 0.4090
⭐ Lưu mô hình tốt nhất...

Epoch 3/12
Train Loss: 0.5544 | Macro F1: 0.4802
Val Loss: 0.6395 | Macro F1: 0.4553
⭐ Lưu mô hình tốt nhất...

Epoch 4/12
Train Loss: 0.4396 | Macro F1: 0.5482
Val Loss: 0.6175 | Macro F1: 0.4792
⭐ Lưu mô hình tốt nhất...

Epoch 5/12
Train Loss: 0.3606 | Macro F1: 0.6065
Val Loss: 0.6172 | Macro F1: 0.5076
⭐ Lưu mô hình tốt nhất...

Epoch 6/12
Train Loss: 0.2996 | Macro F1: 0.6580
Val Loss: 0.6523 | Macro F1: 0.5372
⭐ Lưu mô hình tốt nhất...

Epoch 7/12
Train Loss: 0.2539 | Macro F1: 0.7014
Val Loss: 0.7096 | Macro F1: 0.5639
⭐ Lưu mô hình tốt nhất...

Epoch 8/12
Train Loss: 0.2159 | Macro F1: 0.7410
Val Loss: 0.7519 | Macro F1: 0.5721
⭐ Lưu mô hình tốt nhất...

Epoch 9/12
Train Loss: 0.1866 | Macro F1: 0.7735
Val Loss: 0.7502 | Macro F1: 0.5820
⭐ Lưu mô hình tốt 

In [11]:
import torch
import numpy as np
from sklearn.metrics import f1_score, classification_report

print("--- ĐANG ĐÁNH GIÁ TRÊN TẬP TEST ---")

# Load đúng file đã lưu lúc train
best_path = f"/kaggle/working/saved_models/{model_type}_best.pth"
model.load_state_dict(torch.load(best_path, map_location=device))
print(f"✅ Loaded: {best_path}")

model.eval()
test_targets, test_preds = [], []

with torch.no_grad():
    for data in test_loader:
        input_ids = data["input_ids"].to(device)
        attention_mask = data["attention_mask"].to(device)
        targets = data["targets"].to(device)

        outputs = model(input_ids, attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs[0]

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

        test_preds.extend(preds.cpu().numpy())
        test_targets.extend(targets.cpu().numpy())

test_targets = np.array(test_targets)
test_preds = np.array(test_preds)

macro_f1 = f1_score(test_targets, test_preds, average="macro", zero_division=0)
micro_f1 = f1_score(test_targets, test_preds, average="micro", zero_division=0)

print("\nKẾT QUẢ TRÊN TẬP TEST (best model):")
print(f"🎯 Test Macro F1: {macro_f1:.4f}")
print(f"🎯 Test Micro F1: {micro_f1:.4f}")
print("-" * 50)

# Classification report + lưu file
report = classification_report(
    test_targets, test_preds,
    target_names=list(label_dict.values()),
    zero_division=0,
    output_dict=True,
)
print(classification_report(
    test_targets, test_preds,
    target_names=list(label_dict.values()),
    zero_division=0,
))

pd.DataFrame(report).transpose().to_excel(
    f"/kaggle/working/reports/classification_report_{model_type}_s2.xlsx",
    index=True,
)
print(f"✅ Report saved: /kaggle/working/reports/classification_report_{model_type}_s2.xlsx")

--- ĐANG ĐÁNH GIÁ TRÊN TẬP TEST ---
✅ Loaded: /kaggle/working/saved_models/phobert_best.pth

KẾT QUẢ TRÊN TẬP TEST (best model):
🎯 Test Macro F1: 0.5929
🎯 Test Micro F1: 0.6023
--------------------------------------------------
                precision    recall  f1-score   support

     amusement       0.66      0.87      0.75       374
    excitement       0.43      0.72      0.54        98
           joy       0.43      0.79      0.56       204
          love       0.55      0.86      0.67       143
        desire       0.35      0.70      0.47        80
      optimism       0.60      0.88      0.71       142
        caring       0.47      0.80      0.59       150
         pride       0.65      0.78      0.71        86
    admiration       0.49      0.74      0.59       101
     gratitude       0.73      0.94      0.82       108
        relief       0.43      0.78      0.55        60
      approval       0.47      0.73      0.57       115
   realization       0.35      0.54      0.